In [ ]:
import os
import json
import gzip
import pickle
from glob import glob
from pathlib import Path

import loguru
import numpy as np
import pandas as pd
import open3d as o3d
import torch
from omegaconf import OmegaConf

from moma_sg.data.arti4d import load_arti4d_ground_truth
from moma_sg.utils.mapping import Hierarchy
from moma_sg.utils.metrics import IOU_THRESHOLDS, evaluate_part_segmentation

GT_DATA_ROOT = Path('../input_data/arti4d/raw')
GT_ARTICULATION, _, _ = load_arti4d_ground_truth(GT_DATA_ROOT)


In [ ]:
from hydra import compose, initialize_config_dir

# Hierarchy(cfg) only stores cfg (it is not used by .load()), but we build a real
# config here via Hydra's compose API to mirror how eval_mapping.py's @hydra.main
# entrypoint constructs it.
with initialize_config_dir(config_dir=str(Path('../configs').resolve()), version_base=None):
    cfg = compose(config_name='eval_mapping')

print(OmegaConf.to_yaml(cfg))


In [ ]:
## Load Moma-SG, Pandora, ArtiPoint results here
METHOD_PATHS = {
    "momasg": Path('/path/to/output_momasg_rebuttal/arti4d/'),
}
METHOD_SUBPATHS = {
    "momasg": 'hierarchy.pkl',
}
CONSIDERED_SPLITS = ['rh201', 'rr080', 'din080', 'rh078']

RESULTS = {}
PREDICTIONS = {}
GT_OBJECTS = {}

for room in GT_ARTICULATION.keys():
    if room not in CONSIDERED_SPLITS:
        continue
    if room not in RESULTS:
        RESULTS[room] = {}
        PREDICTIONS[room] = {}
        GT_OBJECTS[room] = {}

    for scene in GT_ARTICULATION[room].keys():
        if scene not in RESULTS[room]:
            RESULTS[room][scene] = {}
            PREDICTIONS[room][scene] = {}
            GT_OBJECTS[room][scene] = []

        print(f'Processing room {room}, scene {scene}')

        gt_semantics_dir = GT_DATA_ROOT / room / scene / "objects"
        gt_scene_objects = []
        for gt_ply_file in os.listdir(gt_semantics_dir):
            if gt_ply_file.endswith(".ply"):
                gt_object_path = os.path.join(gt_semantics_dir, gt_ply_file)
                gt_name = gt_ply_file.split(".")[0]
                gt_scene_objects.append({"pcd": o3d.io.read_point_cloud(gt_object_path), "name": gt_name})
                gt_scene_objects[-1]["bbox"] = gt_scene_objects[-1]["pcd"].get_oriented_bounding_box(robust=True)
                loguru.logger.info(f"Loaded ground truth objects from {gt_object_path}")
        GT_OBJECTS[room][scene] = gt_scene_objects

        for method, method_path in METHOD_PATHS.items():
            PREDICTIONS[room][scene][method] = []
            if method == 'momasg':
                # loading associated articulated objects
                hierarchy_file = method_path / room / scene / METHOD_SUBPATHS[method]
                if not hierarchy_file.exists():
                    loguru.logger.warning(f"Hierarchy file {hierarchy_file} does not exist, skipping.")
                    continue

                hier = Hierarchy(cfg)
                hier.load(hierarchy_file)
                for obj_idx, obj in enumerate(hier.objects):
                    if "model" in obj:
                        print(obj["model"])
                        obj_pcd = o3d.geometry.PointCloud()
                        obj_pcd.points = o3d.utility.Vector3dVector(obj["points"])
                        obj["pcd"] = obj_pcd

                        PREDICTIONS[room][scene][method].append(
                            {
                                "pcd": obj["pcd"],
                                "clip_ft": obj["clip_ft"] if "clip_ft" in obj else None,
                                "conf": 1.0,
                                'bbox': obj_pcd.get_oriented_bounding_box(robust=True),
                            }
                        )

            elif method == 'concept-graphs':
                objects_file = method_path / room / scene / METHOD_SUBPATHS[method]
                with gzip.open(objects_file, 'rb') as f:
                    results = pickle.load(f)
                for obj_dict in results['objects']:
                    pcd = o3d.geometry.PointCloud()
                    pcd.points = o3d.utility.Vector3dVector(obj_dict['pcd_np'])
                    pcd.colors = o3d.utility.Vector3dVector(obj_dict['pcd_color_np'])
                    PREDICTIONS[room][scene][method].append(
                        {
                            "pcd": pcd,
                            "clip_ft": torch.from_numpy(obj_dict['clip_ft']),
                            "conf": np.mean(obj_dict['conf']),
                            'bbox': pcd.get_oriented_bounding_box(robust=True),
                        }
                    )
            elif method == 'pandora':
                hierarchy_file = method_path / room / scene / METHOD_SUBPATHS[method]

                obj_paths = glob(str(hierarchy_file.parent / "object_***_cloud.ply"))
                for obj_path in obj_paths:
                    obj_pcd = o3d.io.read_point_cloud(obj_path)
                    PREDICTIONS[room][scene][method].append(
                        {
                            "pcd": obj_pcd,
                            "clip_ft": None,
                            "conf": 1.0,
                            'bbox': obj_pcd.get_oriented_bounding_box(robust=True),
                        }
                    )
            else:
                raise ValueError(f'Unknown method: "{method}"')

            loguru.logger.info(
                f"Evaluating {method} predictions for room {room}, scene {scene} with "
                f"{len(PREDICTIONS[room][scene][method])} predicted objects and {len(GT_OBJECTS[room][scene])} GT objects."
            )
            RESULTS[room][scene][method] = evaluate_part_segmentation(PREDICTIONS[room][scene][method], GT_OBJECTS[room][scene])


In [7]:
def aggregate_results(RESULTS, method):
    # Pool per-object values (IoUs of matched objects, per-GT-object recall flags)
    # across all rooms/scenes so mean/std are computed across objects, not scenes/rooms.
    all_results = {thresh: {"ious_per_object": [], "recall_per_object": []} for thresh in IOU_THRESHOLDS}
    for room_data in RESULTS.values():
        for scene_data in room_data.values():
            if len(scene_data) == 0:
                continue
            if "tp" in scene_data[method]:
                continue
            for iou, val in scene_data[method].items():
                all_results[iou]["ious_per_object"].extend(val["ious_per_object"])
                all_results[iou]["recall_per_object"].extend(val["recall_per_object"])
    return all_results


for method in METHOD_PATHS.keys():
    print(f"Overall results for method: {method}")
    method_results = aggregate_results(RESULTS, method)
    for thresh in sorted(method_results.keys()):
        ious = method_results[thresh]["ious_per_object"]
        recalls = method_results[thresh]["recall_per_object"]
        mean_iou = np.nanmean(ious) if len(ious) > 0 else 0.0
        std_iou = np.nanstd(ious) if len(ious) > 0 else 0.0
        mean_recall = np.nanmean(recalls) if len(recalls) > 0 else 0.0
        std_recall = np.nanstd(recalls) if len(recalls) > 0 else 0.0
        print(
            f"  mIoU@{thresh}: {mean_iou:.3f} +/- {std_iou:.3f}, "
            f"recall@{thresh}: {mean_recall:.3f} +/- {std_recall:.3f} "
            f"(n={len(ious)} matched objects, {len(recalls)} GT objects)"
        )
    print("\n")


Overall results for method: momasg
  mIoU@0.0: 0.425 +/- 0.261, recall@0.0: 0.997 +/- 0.051 (n=390 matched objects, 391 GT objects)
  mIoU@0.1: 0.511 +/- 0.204, recall@0.1: 0.821 +/- 0.383 (n=321 matched objects, 391 GT objects)
  mIoU@0.2: 0.550 +/- 0.175, recall@0.2: 0.739 +/- 0.439 (n=289 matched objects, 391 GT objects)
  mIoU@0.25: 0.565 +/- 0.163, recall@0.25: 0.706 +/- 0.456 (n=276 matched objects, 391 GT objects)
  mIoU@0.3: 0.588 +/- 0.147, recall@0.3: 0.655 +/- 0.475 (n=256 matched objects, 391 GT objects)
  mIoU@0.4: 0.624 +/- 0.123, recall@0.4: 0.568 +/- 0.495 (n=222 matched objects, 391 GT objects)
  mIoU@0.5: 0.665 +/- 0.102, recall@0.5: 0.455 +/- 0.498 (n=178 matched objects, 391 GT objects)
  mIoU@0.6: 0.717 +/- 0.076, recall@0.6: 0.315 +/- 0.464 (n=123 matched objects, 391 GT objects)
  mIoU@0.7: 0.777 +/- 0.055, recall@0.7: 0.161 +/- 0.368 (n=63 matched objects, 391 GT objects)
  mIoU@0.75: 0.807 +/- 0.046, recall@0.75: 0.102 +/- 0.303 (n=40 matched objects, 391 GT ob

In [8]:
evaluated_method = 'momasg'
report_thresh = 0.5

# Flatten per-scene results at a single IoU threshold into a DataFrame for analysis.
rows = []
for room, room_data in RESULTS.items():
    if room not in CONSIDERED_SPLITS:
        continue
    for scene, scene_data in room_data.items():
        if evaluated_method not in scene_data or len(scene_data[evaluated_method]) == 0:
            continue
        metrics_at_thresh = scene_data[evaluated_method][report_thresh]
        rows.append(
            {
                'room': room,
                'scene': scene,
                'num_gt': len(GT_OBJECTS[room][scene]),
                'num_pred': len(PREDICTIONS[room][scene][evaluated_method]),
                'tp': metrics_at_thresh['tp'],
                'recall': metrics_at_thresh['recall'],
                'iou': metrics_at_thresh['iou'],
            }
        )

df_results = pd.DataFrame(rows)
print(f"Total scenes evaluated: {len(df_results)}")
df_results


Total scenes evaluated: 44


,room,scene,num_gt,num_pred,tp,recall,iou
0,rr080,scene_2025-04-22-11-48-01,9,9,6,0.666667,0.738491
1,rr080,scene_2025-04-17-15-33-44,9,9,4,0.444444,0.621712
2,rr080,scene_2025-04-22-09-58-49,7,8,4,0.571429,0.649312
3,rr080,scene_2025-04-22-11-50-40,8,8,6,0.750000,0.727937
4,rr080,scene_2025-04-22-09-56-24,10,10,5,0.500000,0.583910
5,rr080,scene_2025-04-10-16-05-09,14,14,7,0.500000,0.633372
6,rr080,scene_2025-04-10-13-11-16,17,17,2,0.117647,0.619017
7,rr080,scene_2025-04-22-09-53-49,8,8,6,0.750000,0.682967
8,rr080,scene_2025-04-22-11-45-15,9,9,6,0.666667,0.675576
9,din080,scene_2025-04-11-11-44-32,9,11,3,0.333333,0.610691
